# 🍽️ Restaurant Ratings Analysis
### Task: Analyze Aggregate Rating Distribution & Calculate Average Votes

---

**Objectives:**
1. Analyze the distribution of aggregate ratings across restaurants
2. Determine the most common rating range
3. Calculate the average number of votes received by restaurants

---

## 1. Environment Setup & Library Imports

In [ ]:
# Standard Data Analysis Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot Styling
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print("✅ All libraries imported successfully.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   seaborn : {sns.__version__}")

## 2. Data Loading & Initial Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('dataset.csv')

print(f"📦 Dataset Loaded Successfully")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print()
df.head()

In [ ]:
# Dataset Info
print("📋 Column Overview:")
df.info()

In [ ]:
# Descriptive Statistics for key columns
print("📊 Descriptive Statistics — Aggregate Rating & Votes:")
df[['Aggregate rating', 'Votes']].describe().round(2)

In [ ]:
# Check for missing values
missing = df[['Aggregate rating', 'Votes']].isnull().sum()
print("🔍 Missing Values Check:")
print(missing)
print()
print("✅ No missing values detected in key columns." if missing.sum() == 0 else "⚠️ Missing values found — handle before analysis.")

## 3. Rating Distribution Analysis

In [ ]:
# ── Define Rating Bins ────────────────────────────────────────────────────────
bins   = [0, 1, 2, 3, 4, 5]
labels = ['0–1', '1–2', '2–3', '3–4', '4–5']

df['Rating Range'] = pd.cut(
    df['Aggregate rating'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

rating_dist = df['Rating Range'].value_counts().sort_index()
rating_pct  = (rating_dist / rating_dist.sum() * 100).round(2)

summary_table = pd.DataFrame({
    'Rating Range': rating_dist.index,
    'Count'       : rating_dist.values,
    'Percentage'  : rating_pct.values
}).reset_index(drop=True)

print("📊 Rating Distribution Summary:")
print(summary_table.to_string(index=False))
print()

most_common_range = rating_dist.idxmax()
most_common_count = rating_dist.max()
most_common_pct   = rating_pct[most_common_range]

print(f"🏆 Most Common Rating Range : {most_common_range}")
print(f"   Number of Restaurants   : {most_common_count:,}")
print(f"   Share of Total          : {most_common_pct}%")

In [ ]:
# ── Detailed Rating Text Category Distribution ────────────────────────────────
rating_text_dist = df['Rating text'].value_counts()
print("📋 Rating Category Breakdown (Rating Text):")
print(rating_text_dist.to_string())

## 4. Visualizations

In [ ]:
# ── Figure 1: Histogram of Aggregate Ratings ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    df['Aggregate rating'],
    bins=20,
    color='#4C72B0',
    edgecolor='white',
    linewidth=0.6,
    alpha=0.88
)

mean_rating   = df['Aggregate rating'].mean()
median_rating = df['Aggregate rating'].median()

ax.axvline(mean_rating,   color='#DD3333', linestyle='--', linewidth=1.4, label=f'Mean   : {mean_rating:.2f}')
ax.axvline(median_rating, color='#FF9900', linestyle=':',  linewidth=1.4, label=f'Median : {median_rating:.2f}')

ax.set_title('Distribution of Aggregate Ratings', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Aggregate Rating', fontsize=11)
ax.set_ylabel('Number of Restaurants', fontsize=11)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('fig1_rating_histogram.png', bbox_inches='tight')
plt.show()
print("✅ Figure 1 saved: fig1_rating_histogram.png")

In [ ]:
# ── Figure 2: Bar Chart — Count per Rating Range ─────────────────────────────
colors = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#956CB4']

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    rating_dist.index,
    rating_dist.values,
    color=colors,
    edgecolor='white',
    linewidth=0.7,
    width=0.6
)

for bar, count, pct in zip(bars, rating_dist.values, rating_pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f'{count:,}\n({pct}%)',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

ax.set_title('Restaurant Count by Rating Range', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Rating Range', fontsize=11)
ax.set_ylabel('Number of Restaurants', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('fig2_rating_range_bar.png', bbox_inches='tight')
plt.show()
print("✅ Figure 2 saved: fig2_rating_range_bar.png")

In [ ]:
# ── Figure 3: Pie Chart — Rating Range Share ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

wedges, texts, autotexts = ax.pie(
    rating_dist.values,
    labels=rating_dist.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    pctdistance=0.80,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)

for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')

ax.set_title('Proportional Share of Rating Ranges', fontsize=13, fontweight='bold', pad=16)
plt.tight_layout()
plt.savefig('fig3_rating_pie.png', bbox_inches='tight')
plt.show()
print("✅ Figure 3 saved: fig3_rating_pie.png")

In [ ]:
# ── Figure 4: Rating Text Category Bar ───────────────────────────────────────
order  = ['Not rated', 'Poor', 'Average', 'Good', 'Very Good', 'Excellent']
order  = [o for o in order if o in rating_text_dist.index]
counts = [rating_text_dist[o] for o in order]

palette = sns.color_palette('Blues_d', len(order))

fig, ax = plt.subplots(figsize=(10, 5))
bars2   = ax.barh(order, counts, color=palette, edgecolor='white', linewidth=0.6)

for bar, cnt in zip(bars2, counts):
    ax.text(
        bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
        f'{cnt:,}', va='center', ha='left', fontsize=9, fontweight='bold'
    )

ax.set_title('Restaurant Count by Rating Category', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Restaurants', fontsize=11)
ax.set_ylabel('Rating Category', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('fig4_rating_category.png', bbox_inches='tight')
plt.show()
print("✅ Figure 4 saved: fig4_rating_category.png")

In [ ]:
# ── Figure 5: Box Plot — Votes by Rating Range ────────────────────────────────
active = df[df['Aggregate rating'] > 0].copy()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=active,
    x='Rating Range',
    y='Votes',
    palette='Set2',
    ax=ax,
    order=labels
)

ax.set_title('Vote Distribution Across Rating Ranges', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Rating Range', fontsize=11)
ax.set_ylabel('Number of Votes', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('fig5_votes_boxplot.png', bbox_inches='tight')
plt.show()
print("✅ Figure 5 saved: fig5_votes_boxplot.png")

## 5. Average Votes Calculation

In [ ]:
# ── Overall Average Votes ─────────────────────────────────────────────────────
avg_votes_all    = df['Votes'].mean()
median_votes_all = df['Votes'].median()
total_votes      = df['Votes'].sum()
total_restaurants = len(df)

print("=" * 52)
print("          VOTES ANALYSIS — ALL RESTAURANTS")
print("=" * 52)
print(f"  Total Restaurants       : {total_restaurants:>10,}")
print(f"  Total Votes Cast        : {total_votes:>10,}")
print(f"  Average Votes           : {avg_votes_all:>10.2f}")
print(f"  Median Votes            : {median_votes_all:>10.2f}")
print(f"  Std Deviation           : {df['Votes'].std():>10.2f}")
print(f"  Min Votes               : {df['Votes'].min():>10,}")
print(f"  Max Votes               : {df['Votes'].max():>10,}")
print("=" * 52)

In [ ]:
# ── Average Votes by Rating Range ────────────────────────────────────────────
votes_by_range = df.groupby('Rating Range', observed=True)['Votes'].agg(
    Count='count',
    Average_Votes='mean',
    Median_Votes='median',
    Total_Votes='sum'
).round(2)

print("📊 Average Votes per Rating Range:")
print(votes_by_range.to_string())

In [ ]:
# ── Average Votes by Rating Category ─────────────────────────────────────────
votes_by_cat = df.groupby('Rating text')['Votes'].agg(
    Count='count',
    Average_Votes='mean',
    Median_Votes='median'
).round(2).sort_values('Average_Votes', ascending=False)

print("📊 Average Votes per Rating Category:")
print(votes_by_cat.to_string())

In [ ]:
# ── Figure 6: Average Votes by Rating Range (Bar) ────────────────────────────
avg_by_range = df.groupby('Rating Range', observed=True)['Votes'].mean()

fig, ax = plt.subplots(figsize=(9, 5))
bars3 = ax.bar(
    avg_by_range.index,
    avg_by_range.values,
    color=['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#956CB4'],
    edgecolor='white',
    linewidth=0.7,
    width=0.55
)

ax.axhline(avg_votes_all, color='black', linestyle='--', linewidth=1.2,
           label=f'Overall Average: {avg_votes_all:.1f}')

for bar, val in zip(bars3, avg_by_range.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f'{val:.1f}',
        ha='center', va='bottom', fontsize=9.5, fontweight='bold'
    )

ax.set_title('Average Votes Received per Rating Range', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Rating Range', fontsize=11)
ax.set_ylabel('Average Votes', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig6_avg_votes_bar.png', bbox_inches='tight')
plt.show()
print("✅ Figure 6 saved: fig6_avg_votes_bar.png")

## 6. Key Findings Summary

In [ ]:
print("=" * 60)
print("              KEY FINDINGS SUMMARY")
print("=" * 60)
print()
print("DATASET")
print(f"  Total Restaurants Analyzed : {len(df):,}")
print()
print("RATING DISTRIBUTION")
print(f"  Rating Scale               : 0.0 – 5.0")
print(f"  Mean Rating                : {df['Aggregate rating'].mean():.2f}")
print(f"  Median Rating              : {df['Aggregate rating'].median():.2f}")
print(f"  Most Common Range          : {most_common_range}  ({most_common_count:,} restaurants, {most_common_pct}%)")
print()
print("VOTES")
print(f"  Average Votes per Restaurant: {avg_votes_all:.2f}")
print(f"  Median Votes per Restaurant : {median_votes_all:.2f}")
print(f"  Total Votes in Dataset      : {total_votes:,}")
print()
print("=" * 60)